In [2]:
!pip install torch torchvision torchaudio
    

  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached triton-3.6.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cublas-13.1.0.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime-13.0.96-p

In [1]:
import torch

print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Torch version: 2.11.0+cu130
GPU available: False
Device: cpu


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np
import torch.nn.functional as F

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# --------------------------------------------------
# DEVICE SETUP (AUTO GPU / CPU)
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --------------------------------------------------
# FEDGDA AUGMENTATION (YOUR METHOD)
# --------------------------------------------------
def compute_stats(images):
    mean = images.mean().item()
    std = images.std().item()
    return mean, std


def dynamic_augmentation(images, client_mean, client_std, global_mean, global_std):
    
    # brightness alignment
    gamma = global_mean / (client_mean + 1e-5)
    images = images ** gamma

    # noise adjustment
    noise = torch.randn_like(images) * (global_std - client_std)
    images = images + noise

    return images


# --------------------------------------------------
# SIMPLE MODEL (FOR TESTING)
# --------------------------------------------------
class SimpleUNet(nn.Module):
    def __init__(self):
        super(SimpleUNet, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 5, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.conv3(x)
        return x


# --------------------------------------------------
# DUMMY DATA (replace with your dataset later)
# --------------------------------------------------
batch_size = 4
images = torch.rand(batch_size, 1, 207, 207).to(device)
masks = torch.randint(0, 5, (batch_size, 207, 207)).to(device)


# --------------------------------------------------
# COMPUTE CLIENT STATS (SIMULATION)
# --------------------------------------------------
client_mean, client_std = compute_stats(images)

# simulate global stats (for now)
global_mean = client_mean + 0.1
global_std = client_std + 0.05


# --------------------------------------------------
# APPLY FEDGDA AUGMENTATION
# --------------------------------------------------
images = dynamic_augmentation(images, client_mean, client_std, global_mean, global_std)


# --------------------------------------------------
# MODEL + TRAINING STEP
# --------------------------------------------------
model = SimpleUNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# forward pass
outputs = model(images)

# loss
loss = criterion(outputs, masks)

# backward
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Training step completed. Loss:", loss.item())

Using device: cpu
Training step completed. Loss: 1.61305832862854
